# Treating imbalanced datasets
When performing classification tasks, an important check before training any model is to verify if our classes are balanced (each class set has the same size). Many times, our datasets will be imbalanced most of them the difference among classes is small but sometimes dominant classes having most instances are possible. This is dangerous to our models because on model training, it can identify this trend and introduce a bias tending to classify instances in the majority class just because there were most instances belonging to the class.

**Note:** These strategies **must be performed in training dataset** not in **testing or validation split**.

## Over-sampling techniques
Over sampling techniques works adding instances to the minor classes for equalise class size, some of these strategies are simpler as duplicating rows, another complex approaches generates syntethic data from current features. Let's see some of the mosts used.

### Simple random sampling with replacement
This is the simplest approach, consists on selecting random instances and duplicate them until equalize the amount of instance in majority class. The main advantage is its simplicity but can cause overfitting because we are repeating the same data many times.

In [1]:
from pandas import read_csv

DATA = read_csv("/kaggle/input/datasets/arashnic/imbalanced-data-practice/aug_train.csv")

data = DATA.drop(columns=["id"])

# Bin: Vehicle_Damage, Previously_Insured, Driving_License
CATS = ["Gender", "Driving_License", "Region_Code", "Previously_Insured", "Vehicle_Age", "Vehicle_Damage", "Policy_Sales_Channel"]
NUMS = ["Age", "Annual_Premium", "Vintage"]
TARGET = "Response"

In [12]:
from imblearn.metrics import sensitivity_score, specificity_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

def ml_pipeline(data, sampler = None, enable_sampling = True):
    x_train, x_test, y_train, y_test = train_test_split(data[CATS + NUMS], data[TARGET], test_size=0.2, random_state=123)
    X_train = x_train.copy()
    X_test = x_test.copy()
    
    ENCODER = OrdinalEncoder()
    X_train[["Gender", "Vehicle_Damage", "Vehicle_Age"]] = ENCODER.fit_transform(x_train[["Gender", "Vehicle_Damage", "Vehicle_Age"]])
    X_test[["Gender", "Vehicle_Damage", "Vehicle_Age"]] = ENCODER.transform(x_test[["Gender", "Vehicle_Damage", "Vehicle_Age"]])
    
    if enable_sampling:
        X_train, y_train = sampler.fit_resample(X_train, y_train)

    SCALER = StandardScaler()
    X_train[NUMS] = SCALER.fit_transform(X_train[NUMS])
    X_test[NUMS] = SCALER.transform(X_test[NUMS])

    MODEL = ExtraTreesClassifier(random_state=123, n_jobs=-1)
    MODEL.fit(X_train, y_train)

    PREDS = MODEL.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, PREDS),
        "recall": recall_score(y_test, PREDS),
        "precision": precision_score(y_test, PREDS),
        "f1_score": f1_score(y_test, PREDS),
        "specificity": specificity_score(y_test, PREDS),
        "sensitivity": sensitivity_score(y_test, PREDS),
    }

In [13]:
from imblearn.over_sampling import RandomOverSampler
RANDOM_OVER_SAMPLING_METRICS = ml_pipeline(data, RandomOverSampler(random_state=123))

print(RANDOM_OVER_SAMPLING_METRICS)

{'accuracy': 0.8336146328060604, 'recall': 0.38933522369250156, 'precision': 0.49894014333299685, 'f1_score': 0.4373755696146529, 'specificity': np.float64(0.922115007452734), 'sensitivity': np.float64(0.38933522369250156)}


## SMOTE
Synthentic minority oversampling technique is one of the most popular methods for issuing imbalanced dataset. SMOTE consists on generating new synthetic instances from its neighbors, using interpolation. Procedure consists on:
1. Finding $k$ nearest neighbors to each point beloging to minority class.
2. Choose one of the nearest neighbors of a data point.
3. Select a random point in the straight line that connects the chosen neighbor and the data point. To achieve this goal, interpolation methods are used.
4. Repeat steps 2 and 3 for each neighbor of every data point and then take another data point and continue.

The biggest handicap is the noise generated from synthetic data, resulting in unstable models. Also, generated instances at class bounds are ambiguous. So to issue with some disadvantages and allowing categorical data treatment (simple SMOTE just works on quantitative variables), the following variant were proposed:

- **SMOTE-NC:** This is a variant for working with datasets containing quantitative and qualitative variables.
- **Borderline SMOTE:** Centers on generating "strong" synthetic instances in class bounds.
- **SVM SMOTE:** Sometimes KNN isn't as good as we would like, so there is an alternative that uses a SVM. SVM model is used to find support vectors and generate samples on bound points as borderline SMOTE.
- **SMOTE-N:** A variant when working only with categorical data.

In [15]:
from imblearn.over_sampling import SMOTENC
SMOTE_SAMPLING_METRICS = ml_pipeline(data, SMOTENC(CATS, random_state=123), True)

print(SMOTE_SAMPLING_METRICS)

{'accuracy': 0.8087294422420223, 'recall': 0.7392879647132955, 'precision': 0.45353950229524037, 'f1_score': 0.5621874157707166, 'specificity': np.float64(0.8225621714913313), 'sensitivity': np.float64(0.7392879647132955)}


## ADASYN
This is an approach similar to borderline SMOTE, consists on generating synthetic instances in feature space where the density of minority examples is low, and fewer or none where the density is high. The algorithm is the following.

1. Create a KNN model using the entire dataset.
2. Define a "strenght" coefficient to every point at minority class.
3. Repeat steps 2, 3 and 4 of SMOTE algoritm, based on the calculated strenght coefficient. Focus on generating more instances when coefficient is low.

Key advantage leverages from improving model's performance on instances where is more difficult to differentiate among classes. However, is focused on generating new instances at class bounds and some or any in other parts or feature space.

In [14]:
from imblearn.over_sampling import ADASYN
ADASYN_SAMPLING_METRICS = ml_pipeline(data, ADASYN(random_state=123), True)

print(ADASYN_SAMPLING_METRICS)

{'accuracy': 0.824639217071607, 'recall': 0.5710459987397606, 'precision': 0.47675412638916287, 'f1_score': 0.5196573845106262, 'specificity': np.float64(0.8751549384168824), 'sensitivity': np.float64(0.5710459987397606)}


## Under-sampling strategies
Instead of generating synthetic data, why not removing some instances in the majority class? Under-sampling works removing some samples from every class but minority.

### Random undersampling
Is the opposite of its oversampling counterpart, in this strategy we remove random instances of every class but minority class. The process is repeated after each class has the same sample size. Is a simple and fast way to balance our data, nevertheless, if our minority class size is to small due to model overfitting. Consider that we are removing data with any concern about lossing useful information.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
RANDOM_UNDERSAMPLING_METRICS = ml_pipeline(data, RandomUnderSampler(random_state=123), True)

print(RANDOM_UNDERSAMPLING_METRICS)

### Cluster centroids
Consists on using K-Means algorithm to reduce the number of samples, groups the majority class instances into $K$ clusters ($K$ refers to the size of least class), then finds the regarding centroid of each cluster (centroid is "the most representative" point of a cluster). Finally, centroids are hold and remaining instances are dropped. Algorithms tries to maintain class distribution but useful information can be dropped as we are using the centroid (sometimes can't be the most representative point in a cluster), also centroids aren't necessarily real instances. 

In [ ]:
from imblearn.under_sampling import ClusterCentroids
CLUSTER_UNDERSAMPLING_METRICS = ml_pipeline(data, ClusterCentroids(random_state=123), True)

print(CLUSTER_UNDERSAMPLING_METRICS)

### Near miss
This category has a collection of 3 undersampling methods that selects examples based on the distance of majority class samples to minority class samples.

- **Near miss 1:** Selects samples in the majority class for which the average distance to the `N` closest samples of the minority class is the smallest.
- **Near miss 2:** Selects samples in the majority class for which the average distance to the `N` farthest samples of the minority class is the smallest.
- **Near miss 3:** For each instance in minority class, `M` nearest neighbors in majority class will be kept. Then, samples in majority class are selected for which the average distance to the `N` nearest neighbors is the largest.

Can keep bound samples in majority class but those can produce some noise due to they are lower representative to the class.

In [ ]:
from imblearn.under_sampling import NearMiss
NEAR_MISS_UNDERSAMPLING_METRICS = ml_pipeline(data, NearMiss(version=2), True)

print(NEAR_MISS_UNDERSAMPLING_METRICS)

### Tomek's links
Removes data points from different classes that are closest neighbors to each other, this concept is defined as "Tomek's link". Formerly, a Tomek's link between to samples $x$ and $y$ from different classes is defined for any sample $z$:

<center>
    ($distance(x, y) < distance(x,z)) \wedge (distance(x, y) < distance(y,z))$
</center>
<br />
Underlying idea is removing noisy and hard to classify observations that wouldn't help the model to find a suitable discrimation boundary.

In [ ]:
from imblearn.under_sampling import TomekLinks
TOMEK_UNDERSAMPLING_METRICS = ml_pipeline(data, TomekLinks(n_jobs=-1), True)

print(TOMEK_UNDERSAMPLING_METRICS)